In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

Your Name and PRN:
- Name: ______________________
- PRN : ______________________
- Date: ______________________

# Image Processing with Neural Network

## Assignment A05 :
## Working with PyTorch
- Custom DataSet with common transformation
- Activation function of your choice
- Regularization:
    - L2,
    - BatchNorm,
    - Dropout,
    - Early Stopping.

- multi-class output
- Fashion MNIST dataset

In [ ]:
###-----------------
### Import Libraries
###-----------------

import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from utils.helper import fn_plot_torch_hist, fn_plot_confusion_matrix

In [ ]:
###----------------------
### Some basic parameters
###----------------------

inpDir   = '../../input'
outDir   = '../output'
modelDir = '../models'
subDir   = 'fashion_mnist'
altName  = 'A05_fashion_cnn'

RANDOM_STATE = 24
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

EPOCHS      = 60
BATCH_SIZE  = 64
ALPHA       = 0.0005     # lower LR — reduces val fluctuation
WEIGHT_DECAY = 1e-4      # L2 regularization
PATIENCE    = 15
LR_PATIENCE = 5
LR_FACTOR   = 0.3
MIN_LR      = 1e-7

# parameters for Matplotlib
params = {'legend.fontsize': 'x-large',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'x-large',
          'axes.titlesize': 'x-large',
          'xtick.labelsize': 'x-large',
          'ytick.labelsize': 'x-large'}

CMAP = plt.cm.coolwarm
plt.rcParams.update(params)

In [ ]:
os.makedirs(os.path.join(modelDir, subDir), exist_ok=True)

## All about CUDA

In [ ]:
print('Is CUDA available: ', torch.cuda.is_available())
print('CUDA version:      ', torch.version.cuda)

if torch.cuda.is_available():
    print('Current Device ID: ', torch.cuda.current_device())
    print('Device Name:       ', torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed=RANDOM_STATE)
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using {device} device')

## Load Fashion MNIST Dataset

In [ ]:
class_names = {0: 'T-shirt/top', 1: 'Trouser',  2: 'Pullover', 3: 'Dress', 4: 'Coat',
               5: 'Sandal',      6: 'Shirt',     7: 'Sneaker',  8: 'Bag',   9: 'Ankle boot'}

In [ ]:
local_path = r'D:\Deep_Learning\SharedData\fashion-mnist_train.csv'

if os.path.exists(local_path):
    # ── Local VS Code ──
    inpDir_data = r'D:\Deep_Learning\SharedData'
    train_df = pd.read_csv(os.path.join(inpDir_data, 'fashion-mnist_train.csv'), header=0)
    test_df  = pd.read_csv(os.path.join(inpDir_data, 'fashion-mnist_test.csv'),  header=0)

else:
    # ── Google Colab — load from Keras ──
    import tensorflow as tf
    (X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

    # Name pixel columns as strings to avoid dict sorting conflict
    pixel_cols = [f'p{i}' for i in range(784)]

    train_df = pd.DataFrame(X_train.reshape(-1, 784), columns=pixel_cols)
    train_df.insert(0, 'label', y_train)

    test_df = pd.DataFrame(X_test.reshape(-1, 784), columns=pixel_cols)
    test_df.insert(0, 'label', y_test)

print('Train shape:', train_df.shape)   # (60000, 785)
print('Test shape :', test_df.shape)    # (10000, 785)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

train_df['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], title='Train Label Distribution',
    color='DarkBlue', alpha=0.7)
axes[0].set_xticklabels([class_names[i] for i in range(10)], rotation=45, ha='right')

test_df['label'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], title='Test Label Distribution',
    color='Orange', alpha=0.7)
axes[1].set_xticklabels([class_names[i] for i in range(10)], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Custom Dataset

In [ ]:
class FashionMNISTDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        super(FashionMNISTDataset, self).__init__()

        # labels: first column
        self.labels   = dataframe.iloc[:, 0].to_numpy().astype(np.int64)

        # features: all remaining columns → reshape to (28,28) image
        self.images   = dataframe.iloc[:, 1:].to_numpy().astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # Reshape flat 784 → (28, 28, 1), normalize 0-255 → 0-1
        image = self.images[index].reshape(28, 28, 1) / 255.0
        label = self.labels[index]

        # (H, W, C) → (C, H, W) for PyTorch
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# ── Augmentation for train (active only during training) ──
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))   # randomly erase small patches
])

# ── Test: no augmentation ──
test_transform = None

In [ ]:
train_dataset = FashionMNISTDataset(train_df, transform=train_transform)
test_dataset  = FashionMNISTDataset(test_df,  transform=test_transform)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples : {len(test_dataset)}')

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)

test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

In [ ]:
def show_samples(data_loader, class_labels, num_samples=BATCH_SIZE):
    images, labels = next(iter(data_loader))

    cols = 8
    rows = num_samples // 8

    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 2))
    axes = axes.ravel()

    for i in range(num_samples):
        img = images[i].squeeze().numpy()
        axes[i].imshow(img, cmap=plt.cm.binary)
        axes[i].set_title(class_labels[labels[i].item()], fontsize=7)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
show_samples(train_loader, class_names)

In [ ]:
show_samples(test_loader,  class_names)

## Model Architecture

### Size Flow (28x28 input — Fashion MNIST)
```
Input:  28 x 28 x 1

Set 1: Conv2D(32, 3x3, same) → BN → ReLU → MaxPool(2,2) → Dropout(0.1) → 14 x 14 x 32
Set 2: Conv2D(64, 3x3, same) → BN → ReLU → MaxPool(2,2) → Dropout(0.2) →  7 x  7 x 64
Set 3: Conv2D(128,3x3, same) → BN → ReLU              → Dropout(0.3) →  7 x  7 x 128

Flatten → 7 x 7 x 128 = 6272

Dense(256) → BN → ReLU → Dropout(0.4)
Dense(10)  → logits (CrossEntropyLoss handles softmax)
```
- **BatchNorm** in Conv blocks — normalizes activations, faster convergence
- **Dropout** progressive 0.1→0.2→0.3→0.4 — from CNN reference
- **L2** via AdamW weight_decay=1e-4 — penalizes large weights
- **Early Stopping** — restores best model weights

In [ ]:
class CNNFashion(nn.Module):

    def __init__(self, num_classes=10):
        super(CNNFashion, self).__init__()

        # ── Dropout rates (progressive — from CNN reference) ──
        dor1 = 0.10
        dor2 = 0.20
        dor3 = 0.30
        dor4 = 0.40

        # ── Set 1: 28x28x1 → MaxPool → 14x14x32 ──
        self.conv1    = nn.Conv2d(in_channels=1, out_channels=32,
                                  kernel_size=3, padding='same')
        self.bn1      = nn.BatchNorm2d(32)
        self.relu1    = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)   # 14x14x32
        self.do1      = nn.Dropout(p=dor1)

        # ── Set 2: 14x14x32 → MaxPool → 7x7x64 ──
        self.conv2    = nn.Conv2d(in_channels=32, out_channels=64,
                                  kernel_size=3, padding='same')
        self.bn2      = nn.BatchNorm2d(64)
        self.relu2    = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)   # 7x7x64
        self.do2      = nn.Dropout(p=dor2)

        # ── Set 3: 7x7x64 → (no MaxPool) → 7x7x128 ──
        self.conv3    = nn.Conv2d(in_channels=64, out_channels=128,
                                  kernel_size=3, padding='same')
        self.bn3      = nn.BatchNorm2d(128)
        self.relu3    = nn.ReLU()
        self.do3      = nn.Dropout(p=dor3)

        # ── Flatten: 7 x 7 x 128 = 6272 ──
        self.flatten  = nn.Flatten()

        # ── Dense head ──
        self.fc1      = nn.Linear(6272, 256)
        self.bn_fc1   = nn.BatchNorm1d(256)
        self.relu_fc1 = nn.ReLU()
        self.do4      = nn.Dropout(p=dor4)

        # ── Output — raw logits ──
        self.fc2      = nn.Linear(256, num_classes)

        # ── Weight initialisation (GlorotUniform = Xavier) ──
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Conv blocks
        x = self.do1(self.maxpool1(self.relu1(self.bn1(self.conv1(x)))))
        x = self.do2(self.maxpool2(self.relu2(self.bn2(self.conv2(x)))))
        x = self.do3(self.relu3(self.bn3(self.conv3(x))))

        # Flatten
        x = self.flatten(x)

        # Dense head
        x = self.do4(self.relu_fc1(self.bn_fc1(self.fc1(x))))

        return self.fc2(x)


model = CNNFashion(num_classes=10).to(device)
print(model)

In [ ]:
# ── Loss ──
loss_fn = nn.CrossEntropyLoss()

# ── Optimizer: AdamW — L2 via weight_decay + gradient clipping ──
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=ALPHA,
    weight_decay=WEIGHT_DECAY   # L2 regularization
)

# ── LR Scheduler ──
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LR
)

save_path = os.path.join(modelDir, subDir, f'{altName}.pth')
print('Setup complete.')

## Train & Evaluate Functions

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc  = 0.0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs    = model(inputs)
        batch_loss = loss_fn(outputs, labels)
        batch_loss.backward()

        # Gradient clipping — removes loss spikes
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        preds       = torch.argmax(outputs, dim=1)
        total_loss += batch_loss.item() * inputs.size(0)
        total_acc  += (preds == labels).sum().item()

    return total_loss / len(loader.dataset), total_acc / len(loader.dataset)


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_acc  = 0.0

    with torch.inference_mode():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs    = model(inputs)
            batch_loss = loss_fn(outputs, labels)

            preds       = torch.argmax(outputs, dim=1)
            total_loss += batch_loss.item() * inputs.size(0)
            total_acc  += (preds == labels).sum().item()

    return total_loss / len(loader.dataset), total_acc / len(loader.dataset)

## Training Loop

In [ ]:
n_epoch      = []
train_losses = []
test_losses  = []
train_accs   = []
test_accs    = []

minLoss  = float('inf')
counter  = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss,  test_acc  = evaluate(model, test_loader, loss_fn, device)

    scheduler.step(test_loss)

    n_epoch.append(epoch)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    # ── Early stopping + save best model ──
    if test_loss < minLoss:
        minLoss = test_loss
        counter = 0
        torch.save({'epoch':                epoch + 1,
                    'model_state_dict':     model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss':                 test_loss}, save_path)
    else:
        counter += 1

    if counter > PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Loss: {train_loss:.4f}/{test_loss:.4f} | '
              f'Acc: {train_acc:.4f}/{test_acc:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f} | '
              f'Counter: {counter}')

print(f'\nBest test loss: {minLoss:.4f}')

In [ ]:
loss_df = pd.DataFrame({'epoch':     n_epoch,
                        'loss':      train_losses,
                        'test_loss': test_losses,
                        'acc':       train_accs,
                        'test_acc':  test_accs})

loss_df.head()

In [ ]:
fn_plot_torch_hist(loss_df)

## Training Accuracy

In [ ]:
y_train, y_pred = [], []

model.eval()

with torch.inference_mode():
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds   = torch.argmax(outputs, dim=1)
        y_train.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(f'Train Accuracy: {accuracy_score(y_train, y_pred):.5f}\n')
print(classification_report(y_train, y_pred, target_names=list(class_names.values())))

In [ ]:
fn_plot_confusion_matrix(y_train, y_pred, class_names)

## Testing Accuracy

In [ ]:
y_test, y_pred = [], []

model.eval()

with torch.inference_mode():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        preds   = torch.argmax(outputs, dim=1)
        y_test.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.5f}\n')
print(classification_report(y_test, y_pred, target_names=list(class_names.values())))

In [ ]:
fn_plot_confusion_matrix(y_test, y_pred, class_names)

## Load Best Saved Model

In [ ]:
best_model = CNNFashion(num_classes=10).to(device)
checkpoint = torch.load(save_path)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()
print(f"Loaded best model — epoch {checkpoint['epoch']}, loss {checkpoint['loss']:.4f}")

y_best, y_pred_best = [], []

with torch.inference_mode():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds = torch.argmax(best_model(inputs), dim=1)
        y_best.extend(labels.cpu().numpy())
        y_pred_best.extend(preds.cpu().numpy())

print(f'\nBest Model Test Accuracy: {accuracy_score(y_best, y_pred_best):.5f}\n')
print(classification_report(y_best, y_pred_best, target_names=list(class_names.values())))
fn_plot_confusion_matrix(y_best, y_pred_best, class_names)